# 05. Combining Data from Multiple Sources
**Engineering Data Processing with Python** | one-day course | approx. 45 minutes

> **SOLUTION NOTEBOOK.** Every blank and TODO is filled in. Try the exercise version first; use this when you are stuck or to compare approaches. There is usually more than one correct answer.


In [1]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = ""   # if your trainer gives you a download link for data.zip, paste it between the quotes

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## What you will do

Kilbrack's maintenance history lives in two places: the old spreadsheet (2023 to March 2025) and the CMMS
(January 2025 onwards). During the migration, January to March 2025, many jobs were recorded in **both**.
You will:

1. Align the old spreadsheet's columns and codes to the CMMS schema.
2. Add audit columns so every row says where it came from.
3. Find the jobs recorded in both systems, without a shared ID.
4. Append the two histories into one table.
5. Join on asset details from the register, safely, and find work orders for unknown assets.

**Two operations, two pandas functions:**

| Operation | What it does | pandas |
|---|---|---|
| Append (stack) | More rows, same columns. Jan's export + Feb's export | `pd.concat([a, b])` |
| Join (merge) | More columns, matched on a key. Work order + its asset's type | `a.merge(b, on="key")` |

In [2]:
DATES = ["date_raised", "date_closed", "last_modified"]
cmms = pd.read_csv("data/checkpoints/m04_work_orders_dedup.csv", parse_dates=DATES)
cmms["priority"] = cmms["priority"].astype("Int64")
reg = pd.read_csv("data/checkpoints/m04_asset_register_dedup.csv")
tech_map = pd.read_csv("data/checkpoints/m04_technician_map.csv")

legacy_raw = pd.read_excel("data/work_orders_legacy.xlsx")
decom = pd.read_excel("data/asset_register.xlsx", sheet_name="Decommissioned")
print("CMMS:", cmms.shape, " legacy:", legacy_raw.shape)
legacy_raw.head()

CMMS: (973, 14)  legacy: (494, 9)


,Job No,Equipment,Job Type,Raised,Completed,Hrs,Cost (EUR),Fitter,Notes
0,J10001,PMP-011,Planned,2023-01-06,2023-01-07,5.0,130.51,Niamh O'Connor,Monthly inspection and lubrication
1,J10002,CNC011,Breakdown,2023-01-06,2023-01-08,10.5,152.21,Gráinne Fitzgerald,Tool changer jammed
2,J10003,CNC013,Breakdown,2023-01-06,2023-01-07,4.0,965.19,Orla Doyle,Coolant pump failed
3,J10004,PMP005,Breakdown,2023-01-09,2023-01-12,4.0,208.51,Niamh OConnor,"Bearing noise, drive end"
4,J10005,HYP005,Breakdown,2023-01-09,2023-01-09,7.5,803.80,Patrick Ryan,"Seal failure, leaking"


## 1. Schema alignment

Same information, different column names and codes. Make the legacy table look like the CMMS one.

### Exercise 5.1 [Level 1: fill the blanks]

In [3]:
print(cmms.columns.tolist())
print(legacy_raw.columns.tolist())

['wo_id', 'asset_id', 'wo_type', 'priority', 'date_raised', 'date_closed', 'status', 'labour_hours', 'hours_missing', 'parts_cost_eur', 'technician', 'description', 'last_modified', 'technician_raw']
['Job No', 'Equipment', 'Job Type', 'Raised', 'Completed', 'Hrs', 'Cost (EUR)', 'Fitter', 'Notes']


In [4]:
LEGACY_COLUMNS = {
    "Job No": "wo_id",
    "Equipment": "asset_id",
    "Job Type": "wo_type",
    "Raised": "date_raised",
    "Completed": "date_closed",
    "Hrs": "labour_hours",
    "Cost (EUR)": "parts_cost_eur",
    "Fitter": "technician",
    "Notes": "description",
}
LEGACY_TYPES = {"Planned": "PM", "Breakdown": "CM", "Inspection": "INSP"}

legacy = legacy_raw.rename(columns=LEGACY_COLUMNS)
legacy["wo_type"] = legacy["wo_type"].map(LEGACY_TYPES)
print("Unmapped types:", legacy["wo_type"].isna().sum())
legacy.head(3)

Unmapped types: 0


,wo_id,asset_id,wo_type,date_raised,date_closed,labour_hours,parts_cost_eur,technician,description
0,J10001,PMP-011,PM,2023-01-06,2023-01-07,5.0,130.51,Niamh O'Connor,Monthly inspection and lubrication
1,J10002,CNC011,CM,2023-01-06,2023-01-08,10.5,152.21,Gráinne Fitzgerald,Tool changer jammed
2,J10003,CNC013,CM,2023-01-06,2023-01-07,4.0,965.19,Orla Doyle,Coolant pump failed


Now the details. The asset ID and technician fixes are the same functions as Module 04 (copied here so this
notebook stands on its own).

In [5]:
import unicodedata

def normalise_asset_id(series):
    s = series.astype(str).str.strip().str.upper()
    return s.str.replace(r"^([A-Z]+)-?(\d+)$", r"\1-\2", regex=True)

def normalise_name(name):
    if pd.isna(name):
        return name
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    text = text.replace("'", "").replace(".", " ")
    if "," in text:
        last, first = [part.strip() for part in text.split(",", 1)]
        text = f"{first} {last}"
    return " ".join(text.lower().split())

name_map = dict(zip(tech_map["tech_norm"], tech_map["technician"]))   # reuse Module 04's approved mapping

legacy["asset_id"] = normalise_asset_id(legacy["asset_id"])
legacy["technician_raw"] = legacy["technician"]
legacy["technician"] = legacy["technician"].apply(normalise_name).map(name_map)
print("Technicians not in the approved map:", legacy["technician"].isna().sum())

Technicians not in the approved map: 0


Columns the legacy system never had (`priority`, `last_modified`) stay **missing**. Schema alignment means the same
columns, not invented values. `status` can be derived honestly: a completed date means the job was closed.

In [6]:
legacy["status"] = np.where(legacy["date_closed"].notna(), "Closed", "Open")
legacy["priority"] = pd.Series(pd.NA, index=legacy.index, dtype="Int64")
legacy["last_modified"] = pd.NaT
legacy["hours_missing"] = legacy["labour_hours"].isna()
legacy = legacy[cmms.columns]          # same columns, same order as the CMMS table
legacy.head(3)

,wo_id,asset_id,wo_type,priority,date_raised,date_closed,status,labour_hours,hours_missing,parts_cost_eur,technician,description,last_modified,technician_raw
0,J10001,PMP-011,PM,<NA>,2023-01-06,2023-01-07,Closed,5.0,False,130.51,Niamh O'Connor,Monthly inspection and lubrication,NaT,Niamh O'Connor
1,J10002,CNC-011,CM,<NA>,2023-01-06,2023-01-08,Closed,10.5,False,152.21,Gráinne Fitzgerald,Tool changer jammed,NaT,Gráinne Fitzgerald
2,J10003,CNC-013,CM,<NA>,2023-01-06,2023-01-07,Closed,4.0,False,965.19,Orla Doyle,Coolant pump failed,NaT,Orla Doyle


## 2. Audit columns

When a number in a report looks odd, the first question is "where did this row come from?". Audit columns answer it.

### Exercise 5.2 [Level 1: fill the blanks]

In [7]:
from datetime import datetime
LOADED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")   # one timestamp for the whole run

cmms["source_system"] = "CMMS"
cmms["source_file"] = "work_orders_cmms.csv"
legacy["source_system"] = "LEGACY"
legacy["source_file"] = "work_orders_legacy.xlsx"
for df in (cmms, legacy):
    df["loaded_at"] = LOADED_AT
legacy[["wo_id", "source_system", "source_file", "loaded_at"]].head(3)

,wo_id,source_system,source_file,loaded_at
0,J10001,LEGACY,work_orders_legacy.xlsx,2026-10-06 13:42:46
1,J10002,LEGACY,work_orders_legacy.xlsx,2026-10-06 13:42:46
2,J10003,LEGACY,work_orders_legacy.xlsx,2026-10-06 13:42:46


## 3. Jobs recorded in both systems

The migration jobs have **different IDs** in each system (`J10412` vs `WO-25-00031`), so you cannot match on ID.
Link on what describes the job instead: **same asset, same day raised, same type**. That combination is a
*composite key*. (The legacy sheet only stored the date, not the time, so we compare dates.)

In [8]:
KEY = ["asset_id", "raised_day", "wo_type"]
for df in (cmms, legacy):
    df["raised_day"] = df["date_raised"].dt.normalize()     # drop the time part

window = (legacy["date_raised"] >= "2025-01-06")
print("Legacy jobs inside the migration window:", window.sum())

Legacy jobs inside the migration window: 114


A careful merge states what you expect. `validate="one_to_one"` says "each key appears at most once on each side".
If that is not true, pandas stops with an error instead of multiplying rows behind your back.

In [9]:
try:
    legacy.merge(cmms[KEY], on=KEY, how="left", validate="one_to_one")
except Exception as err:
    print(type(err).__name__, ":", err)

both_sides = cmms[cmms.duplicated(KEY, keep=False)][["wo_id"] + KEY + ["date_raised", "description"]]
both_sides

MergeError : Merge keys are not unique in either left or right dataset; not a one-to-one merge


,wo_id,asset_id,raised_day,wo_type,date_raised,description
58,WO-25-00059,HYP-002,2025-02-11,CM,2025-02-11 08:30:00,Hydraulic leak at cylinder seal
60,WO-25-00061,HYP-002,2025-02-11,CM,2025-02-11 15:10:00,"Seal failure, leaking"
138,WO-25-00139,CMP-007,2025-03-30,PM,2025-03-30 07:10:00,"Grease bearings, check alignment"
139,WO-25-00140,CMP-007,2025-03-30,PM,2025-03-30 07:30:00,Annual service per OEM schedule
297,WO-25-00298,CNC-007,2025-07-12,PM,2025-07-12 08:20:00,"Grease bearings, check alignment"
299,WO-25-00300,CNC-007,2025-07-12,PM,2025-07-12 10:00:00,Annual service per OEM schedule
495,WO-25-00496,CHL-001,2025-11-04,INSP,2025-11-04 09:10:00,Guarding inspection
496,WO-25-00497,CHL-001,2025-11-04,INSP,2025-11-04 18:50:00,Vibration route survey
720,WO-26-00721,ROB-002,2026-03-18,PM,2026-03-18 08:50:00,Quarterly PM service
721,WO-26-00722,ROB-002,2026-03-18,PM,2026-03-18 09:20:00,"Replace filters, check belts"


Good thing we asked. HYP-002 had **two** corrective jobs on 11 Feb 2025. Without `validate`, each legacy job would
match both CMMS jobs and you would get 4 rows instead of 2.

**Fix:** number repeated keys on each side (1st job that day, 2nd job that day) with `groupby().cumcount()` and
include that number in the key.

### Exercise 5.3 [Level 2: finish the code]

In [10]:
cmms = cmms.sort_values("date_raised")
legacy = legacy.sort_values(["date_raised", "wo_id"])
for df in (cmms, legacy):
    df["seq"] = df.groupby(KEY).cumcount()
KEY_SEQ = KEY + ["seq"]

linked = legacy.merge(cmms[KEY_SEQ], on=KEY_SEQ, how="left", validate="one_to_one", indicator=True)
in_both = linked[linked["_merge"] == "both"]
legacy_new = legacy[~legacy["wo_id"].isin(in_both["wo_id"])]

print("Legacy jobs already in CMMS:", len(in_both))
print("Legacy jobs to add:", len(legacy_new))

Legacy jobs already in CMMS: 101
Legacy jobs to add: 393


In [11]:
outside = in_both["date_raised"] < "2025-01-06"
check(len(in_both) > 0 and not outside.any() and len(legacy_new) + len(in_both) == len(legacy),
      "Overlap found, all of it inside the migration window. That is a good sign the key is right.",
      "Expected matches only inside Jan to Mar 2025, and new + overlap = all legacy rows.")

PASS: Overlap found, all of it inside the migration window. That is a good sign the key is right.


That last check is the important one. If the key matched jobs from 2023, it would be too loose (matching different
jobs). Overlap only where we expect it is evidence the linkage is behaving.

Also look at the legacy jobs **inside** the window that did *not* match. Real jobs only in the old system, or the
same job recorded with a different date or type? This is where you would sample a few and check with the
maintenance planner.

In [12]:
legacy_new[legacy_new["date_raised"] >= "2025-01-06"][["wo_id", "asset_id", "wo_type", "date_raised", "description"]].head()

,wo_id,asset_id,wo_type,date_raised,description
384,J10385,CNV-003,CM,2025-01-10,"Belt snapped, replaced"
385,J10386,CNV-009,INSP,2025-01-12,Thermographic survey
405,J10406,HYP-006,PM,2025-01-29,Quarterly PM service
411,J10412,CHL-001,PM,2025-02-04,Monthly inspection and lubrication
419,J10420,PMP-009,CM,2025-02-07,"Bearing noise, drive end"


## 4. Append

### Exercise 5.4 [Level 1: fill the blanks]

In [13]:
combined = pd.concat([cmms, legacy_new], ignore_index=True)
combined = combined.drop(columns=["raised_day", "seq"])
print(len(cmms), "+", len(legacy_new), "=", len(combined))
combined["source_system"].value_counts()

973 + 393 = 1366


source_system
CMMS      973
LEGACY    393
Name: count, dtype: int64

In [14]:
check(len(combined) == len(cmms) + len(legacy_new) and combined["wo_id"].is_unique,
      "Appended, and every work order ID is still unique.", "Row count or uniqueness is off.")

PASS: Appended, and every work order ID is still unique.


## 5. Join on asset details

Work orders need asset type and site for any useful report. Some work orders are on assets that were
decommissioned (they are in the other sheet) and a few reference assets nobody has heard of.

### Exercise 5.5 [Level 2: finish the code]
1. Build one asset lookup from the in-service register **and** the decommissioned sheet (another append with
   schema alignment).
2. Left-join it onto `combined` with `validate="many_to_one"` and `indicator=True`.
3. Label rows with no matching asset as `"Unknown asset"`.
4. Check the row count did not change. (A join that adds rows is the commonest silent bug in data work.)

In [15]:
in_service = reg[["asset_id", "asset_type", "site", "criticality", "needs_review"]].assign(asset_status="In Service")
retired = (decom.rename(columns={"Asset ID": "asset_id", "Asset Type": "asset_type", "Site": "site", "Criticality": "criticality"})
                [["asset_id", "asset_type", "site", "criticality"]]
                .assign(needs_review=False, asset_status="Decommissioned"))
assets_all = pd.concat([in_service, retired], ignore_index=True)
print(len(assets_all), "assets in the lookup, unique:", assets_all["asset_id"].is_unique)

65 assets in the lookup, unique: True


In [16]:
rows_before = len(combined)

wo_full = combined.merge(assets_all, on="asset_id", how="left", validate="many_to_one", indicator=True)
wo_full.loc[wo_full["_merge"] == "left_only", "asset_status"] = "Unknown asset"

wo_full = wo_full.drop(columns="_merge")
print("Rows before:", rows_before, " after:", len(wo_full))
wo_full["asset_status"].value_counts(dropna=False)

Rows before: 1366  after: 1366


asset_status
In Service        1321
Decommissioned      39
Unknown asset        6
Name: count, dtype: int64

In [17]:
check("asset_status" in wo_full and len(wo_full) == rows_before and wo_full["asset_status"].notna().all(),
      "Joined without adding rows. Every work order has an asset status.",
      "Row count changed, or some rows have no asset_status.")

PASS: Joined without adding rows. Every work order has an asset status.


In [18]:
wo_full.loc[wo_full["asset_status"] == "Unknown asset", ["wo_id", "asset_id", "date_raised", "description", "source_system"]]

,wo_id,asset_id,date_raised,description,source_system
35,WO-25-00035,CNV-021,2025-01-29 11:50:00,Quarterly PM service,CMMS
154,WO-25-00155,PMP-020,2025-04-06 16:45:00,"Grease bearings, check alignment",CMMS
475,WO-25-00476,CNC-099,2025-10-24 15:50:00,"Grease bearings, check alignment",CMMS
746,WO-26-00747,CNC-099,2026-04-03 13:50:00,Quarterly PM service,CMMS
790,WO-26-00791,ROB-007,2026-04-28 16:05:00,Hydraulic leak at cylinder seal,CMMS
840,WO-26-00841,CMP-0O1,2026-06-01 15:50:00,Motor tripped on overload,CMMS


Look at the unknown IDs. `CMP-0O1` has a letter **O** instead of a zero. `CNC-099` is probably a typo. Others may be
assets that were never registered. We keep these rows and flag them; Module 06 turns that into a quality rule.

## 6. Save

In [19]:
wo_full.to_csv("outputs/work_orders_combined.csv", index=False)
print(wo_full.shape)
wo_full.groupby(["source_system", "asset_status"]).size()

(1366, 22)


source_system  asset_status  
CMMS           Decommissioned     10
               In Service        957
               Unknown asset       6
LEGACY         Decommissioned     29
               In Service        364
dtype: int64

## Stretch [Level 3: Gemini allowed]  Stacking sheets with different layouts

`thermal-profiles.xlsx` has three alloys on three sheets. One has 7 thermocouples, two have 8, and the positions
differ. You cannot simply stack them side by side.

Build one **long** table with columns `alloy`, `thermocouple`, `position_mm`, `time_s`, `temp_c` for all three
sheets. Hint: load each sheet as in Module 02, then `melt()` it from wide to long, add the alloy name, and `concat`.

**Prompt to try:**
> I have an Excel file with 3 sheets. Each has a title row, a blank row, a row of thermocouple positions in mm, then
> a header row with 'Time [s]' and 'Thermopar #1 [°C]' etc. Sheets have different numbers of thermocouples. Write
> pandas code to combine them into one long table with columns alloy, thermocouple, position_mm, time_s, temp_c.

**Check:** the total row count should equal 2001 x (7 + 8 + 8).

In [20]:
frames = []
for sheet in pd.ExcelFile("data/thermal-profiles.xlsx").sheet_names:
    data = pd.read_excel("data/thermal-profiles.xlsx", sheet_name=sheet, header=3)
    positions = pd.read_excel("data/thermal-profiles.xlsx", sheet_name=sheet, header=None, nrows=3).iloc[2, 1:].tolist()
    data = data.rename(columns={data.columns[0]: "time_s"})
    pos_lookup = dict(zip(data.columns[1:], positions))
    long = data.melt(id_vars="time_s", var_name="thermocouple", value_name="temp_c")
    long["position_mm"] = long["thermocouple"].map(pos_lookup)
    long["alloy"] = sheet
    frames.append(long)
thermal_long = pd.concat(frames, ignore_index=True)[["alloy", "thermocouple", "position_mm", "time_s", "temp_c"]]
print(len(thermal_long), "rows; expected", 2001 * (7 + 8 + 8))
thermal_long.head()

46023 rows; expected 46023


,alloy,thermocouple,position_mm,time_s,temp_c
0,Al-11wt.%Si,Thermopar #1 [°C],4.0,0.0,634.0735
1,Al-11wt.%Si,Thermopar #1 [°C],4.0,0.2,631.0739
2,Al-11wt.%Si,Thermopar #1 [°C],4.0,0.4,625.9844
3,Al-11wt.%Si,Thermopar #1 [°C],4.0,0.6,620.2867
4,Al-11wt.%Si,Thermopar #1 [°C],4.0,0.8,615.1943


## Summary

* Align schemas with a rename dictionary and a code map. Missing columns stay missing.
* Add `source_system`, `source_file`, `loaded_at` to everything you combine.
* No shared ID? Link on a composite key, and check the matches appear only where they should.
* `validate=` turns silent row multiplication into a clear error. Use it on every merge.
* Count rows before and after every join.

Next: **06 Data Quality Controls**.